# Part C — SageMaker fine-tuning, metrics, ClearML

**Course:** 42028 Deep Learning · **Deliverable:** Intermediate 2 (Part C per [`docs/PRD.md`](../../docs/PRD.md))

**Primary path:** **SageMaker Notebook / Studio + EBS** — sync only **`data/processed/splits/`** + **`train/`** (not the full ~22GB repo). Run [`train/scripts/train_yolo.py`](../scripts/train_yolo.py); see [`docs/DATA.md`](../../docs/DATA.md).

1. **Notebook GPU (ml.g4dn.xlarge or local CUDA)** — same CLI; data at repo `data/processed/splits/`.
2. **ClearML** — optional; use [`train/src/utils/clearml_setup.py`](../src/utils/clearml_setup.py).
3. **Managed Training Job (optional)** — S3 upload + [`infra/sagemaker/sagemaker_launch.py`](../../infra/sagemaker/sagemaker_launch.py); helpers: [`train/src/training/sagemaker_cli.py`](../src/training/sagemaker_cli.py).

**Artifacts:** `best.pt`, metrics, hyperparameters for [`docs/reports/Final_Training_Report.md`](../../docs/reports/Final_Training_Report.md).

**Imports:** use the public [`src.training`](../src/training/__init__.py) package (presets, subprocess runner, SageMaker argv builder).

## 0. Environment

Run with kernel **Python 3.10+**. Working directory for code cells below should be **`train/`** (use `%cd ..` from `train/notebooks/` if needed).

```bash
# From repo root (optional venv)
pip install -r requirements.txt
```

In [ ]:
%pip install python-dotenv clearml -q

import os
import sys
from pathlib import Path

# Ensure imports resolve: notebook in train/notebooks -> add train/ to path
train_dir = Path.cwd().resolve()
if train_dir.name == "notebooks":
    train_dir = train_dir.parent
if str(train_dir) not in sys.path:
    sys.path.insert(0, str(train_dir))

os.chdir(train_dir)
print("cwd:", train_dir)

In [ ]:
from src.repo_paths import check_splits_layout, load_repo_dotenv, repo_root
from src.training import build_sagemaker_launch_command, default_training_presets

root = repo_root()
load_repo_dotenv()
print("repo_root:", root)
print("CLEARML keys set:", bool(os.environ.get("CLEARML_API_ACCESS_KEY")))

## 1. Data layout check

On-disk layout for **local / SageMaker Notebook** (no S3): `data/processed/splits/{train,val,test}/{images,labels}/` + `data.yaml`.

In [ ]:
ok, issues = check_splits_layout(root=root)
if ok:
    print("OK: splits layout found.")
else:
    print("Fix data before training:")
    for line in issues:
        print(" -", line)

## 2. Part C hyperparameter presets

| Preset | Use case |
|--------|----------|
| `g4dn_notebook` | SageMaker Notebook **ml.g4dn.xlarge** (or local T4-class) — defaults match README |
| `sagemaker_managed_job` | **Managed training** — larger batch / `yolov8l` typical |

Edit fields below for your Part C tuning grid.

In [ ]:
presets = default_training_presets()
hp = presets["g4dn_notebook"]  # or presets["sagemaker_managed_job"]
print(hp)
print(hp.as_dict())

## 3. Optional: ClearML task

Uncomment after `clearml-init` (or hosted ClearML). Offline: `CLEARML_OFFLINE_MODE=1`.

In [ ]:
# from src.utils.clearml_setup import ClearMLSetup
# cm = ClearMLSetup("CrowdNav", "part_c_yolo_tune")
# cm.init_clearml_task(tags=["part-c", "yolov8"], params=hp.as_dict())
print("ClearML block commented out by default.")

## 4. Run training — local or SageMaker Notebook

Same as [`01` intro](01_sagemaker_clearml_launcher.ipynb): run from **`train/`**.

**Option A — shell (copy/paste):**

```bash
python scripts/train_yolo.py \
  --model-cfg yolov8m.pt --epochs 100 --batch 16 --workers 4 \
  --name crowdnav_part_c
```

**Option B — Python subprocess** (uses preset object):

In [ ]:
from src.training import run_train_yolo_subprocess

# Set True to actually train (long-running)
RUN_LOCAL = False

if RUN_LOCAL:
    code = run_train_yolo_subprocess(
        hyperparams=hp,
        run_name="crowdnav_part_c_notebook",
    )
    print("exit code:", code)
else:
    print("Set RUN_LOCAL = True to execute train_yolo.py from this notebook.")

## 5. Managed SageMaker Training Job (from your laptop)

1. Upload `data/processed/splits/` contents to `s3://YOUR_BUCKET/processed/splits/` (folders `train`, `val`, `test`).
2. IAM role ARN with SageMaker + S3 access.
3. Install: `pip install sagemaker boto3`.
4. Run the printed command from **repository root** (or adjust paths).

Outputs: `best.pt` and metrics under the job’s S3 `output_path` (see [`sagemaker_train.py`](../../infra/sagemaker/sagemaker_train.py)).

In [ ]:
import shlex

# --- set these before running on a machine with AWS CLI / boto3 configured ---
SAGEMAKER_ROLE_ARN = os.environ.get("CROWDNAV_SM_ROLE", "arn:aws:iam::ACCOUNT_ID:role/SageMakerExecutionRole")
SAGEMAKER_BUCKET = os.environ.get("CROWDNAV_SM_BUCKET", "your-jrdb-bucket")

cmd = build_sagemaker_launch_command(
    repo_root=root,
    role_arn=SAGEMAKER_ROLE_ARN,
    bucket=SAGEMAKER_BUCKET,
    data_prefix="processed/splits",
    instance_type="ml.g4dn.xlarge",
    hyperparams=presets["sagemaker_managed_job"],
    use_spot=False,
)
print("Shell:")
print(" ".join(shlex.quote(c) for c in cmd))

## 6. Part C checklist (for the report)

- [ ] Baseline / tuned run on **SageMaker Notebook (EBS)** or managed job with logged **epochs, batch, model, mAP**.
- [ ] **Proximity heuristic** calibration: [`train/src/inference/collision_avoidance.py`](../src/inference/collision_avoidance.py) (reference); production mapping lives in Spring `CrowdNavHeuristics`.
- [ ] **Deploy path:** export `best.pt` → ONNX via [`train/scripts/eval_yolo.py`](../scripts/eval_yolo.py) (`--export-onnx`); run Spring `crowdnav-api` with `app.inference.mode=onnx` — [`docs/runbooks/post_train_spring_onnx.md`](../../docs/runbooks/post_train_spring_onnx.md).

CI: [`.github/workflows/train-pipeline.yml`](../../.github/workflows/train-pipeline.yml) on relevant branches/paths (ruff, mypy, `py_compile`).